---



# Unit 2 - Part 3a: Chain of Thought (CoT)

## 1. Introduction: The Inner Monologue

Standard LLMs try to jump straight to the answer. For complex problems (math, logic), this often fails.

**Chain of Thought (CoT)** forces the model to "think out loud" before answering.

### Why use a "Dumb" Model?
For this unit, we will use **Llama3.1-8b** (via Groq). It is a smaller, faster model.
Why? Because huge models (like Gemini Pro or GPT-4) are often *too smart*—they solve logic riddles instantly without thinking.

To really see the power of Prompt Engineering, we need a model that **needs help**.

### Visualizing the Process (Flowchart)
```mermaid
graph TD
    Input[Question: 5+5*2?]
    Input -->|Standard| Wrong[Answer: 20 (Wrong)]
    Input -->|CoT| Step1[Step 1: 5*2=10]
    Step1 --> Step2[Step 2: 5+10=15]
    Step2 --> Correct[Answer: 15 (Correct)]
```

## 2. Concept: Latent Reasoning

Why does this work?
Because LLMs are "Next Token Predictors".
- If you force it to answer immediately, it must predict the digits `1` and `5` immediately.
- If you let it "think", it generates intermediate tokens (`5`, `*`, `2`, `=`, `1`, `0`).
- The model then **ATTENDS** to these new tokens to compute the final answer.

**Writing is Thinking.**

In [ ]:
# Setup
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Using Llama3.1-8b (Small/Fast) to demonstrate logic failures
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 6.4 MB/s eta 0:00:00
Enter your Groq API Key: ··········


## 3. The Experiment: A Tricky Math Problem

Let's try a problem that requires multi-step logic.

**Problem:**
"Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many does he have now?"

In [ ]:
question = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many does he have now?"

# 1. Standard Prompt (Direct Answer)
prompt_standard = f"Answer this question: {question}"
print("--- STANDARD (Llama3.1-8b) ---")
print(llm.invoke(prompt_standard).content)

--- STANDARD (Llama3.1-8b) ---
To find out how many tennis balls Roger has now, we need to add the initial number of tennis balls he had (5) to the number of tennis balls he bought (2 cans * 3 tennis balls per can).

2 cans * 3 tennis balls per can = 6 tennis balls

Now, let's add the initial number of tennis balls (5) to the number of tennis balls he bought (6):

5 + 6 = 11

So, Roger now has 11 tennis balls.


### Critique
Smaller models often latch onto the visible numbers (5 and 2) and simply add them (7), ignoring the multiplication step implied by "cans".

Let's force it to think.

In [ ]:
# 2. CoT Prompt (Magic Phrase)
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("--- Chain of Thought (Llama3.1-8b) ---")
print(llm.invoke(prompt_cot).content)

--- Chain of Thought (Llama3.1-8b) ---
To find out how many tennis balls Roger has now, we need to follow these steps:

1. Roger already has 5 tennis balls.
2. He buys 2 more cans of tennis balls. Each can has 3 tennis balls, so he buys 2 x 3 = 6 more tennis balls.
3. Now, we add the tennis balls he already had (5) to the new tennis balls he bought (6). 5 + 6 = 11

So, Roger now has 11 tennis balls.


## 4. Analysis

Look at the output. By explicitly breaking it down:
1.  "Roger starts with 5."
2.  "2 cans * 3 balls = 6 balls."
3.  "5 + 6 = 11."

The model effectively "debugs" its own logic by generating the intermediate steps.

---



# Unit 2 - Part 3b: Tree of Thoughts (ToT) & Graph of Thoughts (GoT)

## 1. Introduction: Beyond A -> B

CoT is linear. But complex reasoning is often nonlinear. We need to explore branches (ToT) or even combine ideas (GoT).

We continue using **Llama3.1-8b via Groq** to show how structure improves performance.

In [ ]:
# Setup
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Using Llama3.1-8b
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7) # Creativity needed

## 2. Tree of Thoughts (ToT)

ToT explores multiple branches before making a decision.
**Analogy:** A chess player considering 3 possible moves before playing one.

### Implementation
We will generate 3 distinct solutions for a problem and then use a "Judge" to pick the best one.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "How can I get my 5-year-old to eat vegetables?"

# Step 1: The Branch Generator
prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

# Step 2: The Judge
prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'

    1: {sol1}
    2: {sol2}
    3: {sol3}

    Act as a Child Psychologist. Pick the most sustainable one (not bribery) and explain why.
    """
)

# Chain: Input -> Branches -> Judge -> Output
tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- Tree of Thoughts (ToT) Result ---")
print(tot_chain.invoke(problem))

--- Tree of Thoughts (ToT) Result ---
As a Child Psychologist, I would recommend **Solution 1: Create a "Veggie Face" on Their Plate** as the most sustainable approach to encourage a 5-year-old to eat vegetables. Here's why:

1. **Involves the child in the process**: By creating a "Veggie Face" together, you're involving your child in the meal planning and preparation process. This can help them feel more invested in trying new foods, including vegetables.
2. **Encourages creativity and play**: The "Veggie Face" approach combines creativity, imagination, and play, which are essential for children's cognitive and emotional development. This can make mealtime a more enjoyable and interactive experience for your child.
3. **Builds confidence and self-efficacy**: By creating a "Veggie Face" together, your child will feel more confident and self-efficacious about trying new foods. This can help them develop a sense of control and agency over their food choices.
4. **Fosters a positive relat

## 3. Graph of Thoughts (GoT)

You asked: **"Where is Graph of Thoughts?"**

GoT is more complex. It's a network. Information can split, process specific parts, and then **AGGREGATE** back together.

### The Workflow (Writer's Room)
1.  **Split:** Generate 3 independent story plots (Sci-Fi, Fantasy, Mystery).
2.  **Aggregate:** The model reads all 3 and creates a "Master Plot" that combines the best elements of each.
3.  **Refine:** Polish the Master Plot.

```mermaid
graph LR
   Start(Concept) --> A[Draft 1]
   Start --> B[Draft 2]
   Start --> C[Draft 3]
   A & B & C --> Mixer[Aggregator]
   Mixer --> Final[Final Story]
```

In [ ]:
# 1. The Generator (Divergence)
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror") | llm | StrOutputParser(),
)

# 2. The Aggregator (Convergence)
prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':
    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Horror: {draft_horror}

    Your task: Create a new Mega-Movie that combines the TECHNOLOGY of Sci-Fi, the PASSION of Romance, and the FEAR of Horror.
    Write one paragraph.
    """
)

# 3. The Chain
got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- Graph of Thoughts (GoT) Result ---")
print(got_chain.invoke("Time Travel"))

--- Graph of Thoughts (GoT) Result ---
In "Echoes of Eternity," a brilliant but reclusive physicist, Emma, discovers a way to send her consciousness back in time using an ancient, mysterious watch that's been passed down through her family for generations. As she navigates a series of events in her past to prevent a catastrophic global disaster that will erase her entire family from existence, she finds herself torn between her present-day fiancé, a kind-hearted but increasingly distant partner named Ryan, and a charismatic stranger, Alexander, who appears in the past and captures her heart. However, their blossoming romance is short-lived, as Emma soon realizes that her actions in the past are being manipulated by a malevolent entity, a vengeful spirit that's been awakened by her time traveling, and it's not just her family's fate that's at stake - it's the very fabric of time itself.


### Latent Reasoning Summary

**Latent Reasoning** refers to the idea that LLMs, being 'Next Token Predictors', generate intermediate tokens (like steps in a calculation) that they then 'attend' to. This process of externalizing intermediate thoughts allows the model to effectively 'debug' its own logic and arrive at a more accurate final answer.

### Chain of Thought (CoT) Example

**Example:** The problem was: "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many does he have now?"

Using the CoT prompt: `Answer this question. Let's think step by step. {question}`

Llama3.1-8b's CoT breakdown:
1.  Roger already has 5 tennis balls.
2.  He buys 2 x 3 = 6 more tennis balls.
3.  Now, 5 + 6 = 11.

**CoT Example Summary:** By explicitly prompting the model to "think step by step," the LLM generated a clear, logical sequence of operations (multiplication first, then addition). This process allowed the model to correctly solve a multi-step problem that it might otherwise struggle with if asked for a direct answer, demonstrating how writing out steps aids in the model's internal reasoning and accuracy.

## Notebook Summary: Advanced Prompt Engineering Techniques

This notebook explores advanced prompt engineering techniques to improve the reasoning capabilities of Large Language Models (LLMs), especially with smaller models like Llama3.1-8b. The core concept behind these techniques is **Latent Reasoning**, where LLMs externalize their intermediate thoughts, allowing them to 'attend' to these steps and self-correct their logic.

### 1. Chain of Thought (CoT)

**Concept:** CoT encourages LLMs to 'think step by step' before providing a final answer. This linear reasoning process is crucial for complex problems (e.g., math, logic) where direct answers often fail. By prompting the model to break down the problem, it generates intermediate tokens that aid in more accurate final predictions.

**Example:** Solving a multi-step math problem by explicitly asking the LLM to outline each calculation step (e.g., first multiplication, then addition).

### 2. Tree of Thoughts (ToT)

**Concept:** ToT extends CoT by exploring multiple distinct reasoning paths or solutions (branches) for a given problem. Instead of a single linear chain, ToT generates several potential answers and then uses a 'judge' or evaluation mechanism to select the most suitable one.

**Example:** Generating three unique solutions to a problem (e.g., "How to get a 5-year-old to eat vegetables?") and then having a separate prompt evaluate and choose the best, most sustainable option.

### 3. Graph of Thoughts (GoT)

**Concept:** GoT is the most complex, allowing for non-linear, networked reasoning. It enables information to split, process specific aspects independently, and then aggregate or combine ideas back together. This resembles a creative workflow where different ideas are generated and then merged into a cohesive output.

**Example:** Creating a 'Mega-Movie' plot by generating separate plot ideas (Sci-Fi, Romance, Horror) and then combining the best elements of each into a single, comprehensive narrative.

### Summary Comparison

| Method        | Structure               | Best For...                               | Cost/Latency |
|---------------|-------------------------|-------------------------------------------|--------------|
| **Simple**    | Input -> Output         | Simple facts, summaries                   | Low          |
| **CoT**       | Input -> Steps -> Output| Math, Logic, Debugging                    | Medium       |
| **ToT**       | Input -> Branches -> Select | Strategic decisions, Brainstorming        | High         |
| **GoT**       | Input -> Branch -> Mix/Aggregate | Creative Writing, Research Synthesis      | Very High    |

The recommendation is to start with CoT and escalate to ToT or GoT only for more complex tasks requiring deeper, more structured reasoning.

## 4. Summary & Comparison Table

| Method | Structure | Best For... | Cost/Latency |
|--------|-----------|-------------|--------------|
| **Simple Prompt** | Input -> Output | Simple facts, summaries | ⭐ Low |
| **CoT (Chain)** | Input -> Steps -> Output | Math, Logic, Debugging | ⭐⭐ Med |
| **ToT (Tree)** | Input -> 3x Branches -> Select -> Output | Strategic decisions, Brainstorming | ⭐⭐⭐ High |
| **GoT (Graph)** | Input -> Branch -> Mix/Aggregate -> Output | Creative Writing, Research Synthesis | ⭐⭐⭐⭐ V. High |

**Recommendation:** Start with CoT. Only use ToT/GoT if CoT fails.